# Extraction and Preprocessing 

## What it does: 
Processes raw precise point positioning data from GPS position files to calculate daily ice velocity estimates for the GPS stations. It uses this velcotiy to create a velocity time-series which allows every icequake to have velocity as a feature. 

## Inputs: 
- PPP files 
- Slip event catalog 

## Outputs: 
- Event catalog that gives the pre-event velocity median for each event, the velocity components per station, and how many observations were used 
- Daily velocity time series 

## References: 
- Katz, Z. S., Siegfried, M. R., & Padman, L. (2026). Slip-event timing and ice velocity vary at long-period ocean tidal frequencies at Whillans Ice Plain, West Antarctica. Journal of Geophysical Research: Earth Surface, 131, e2025JF008770. https://doi.org/10.1029/2025JF008770
- https://zenodo.org/records/17797751 
- https://github.com/zsk4/WhillansCatalogPaper/releases/tag/v1.2.1 



In [2]:
import os, glob
import numpy as np
import pandas as pd
from pyproj import Transformer

# path configuration 

PPP_dir = "../Data/PPP/PPP/"
Events_csv = "../Data/whillans_velocity_per_event.csv"
Daily_timeseries_csv = "../Data/ppp_velocity_timeseries.csv"
Event_w_vel_csv = "../Data/whillans_events_with_velocity.csv"

#cut offs for preprocessing/filtering the data 

position_threshold = 0.05    # drop any GPS epochs with RMSP > 5 cm
phys_vel_threshold = 2000    # drop computed velocity that is higher than this (physical upper bound for Antarctic ice streams)
sec_per_year = 365.25 * 86400 # converting m/s to m/yr
min_epochs_day = 1000    # setting a limit for how many epochs are needed to give a reasonable median (this is about 17% so a low threshold)
max_day_gap = 2       # max days between consecutive daily estimates
pre_window_days = 30      # days before event to average for pre-event velocity
min_daily_pts = 5       # minimum daily estimates in pre-event window

# Cooridnate Projection
# lat/lon to EPSG:3031 (Antarctic polar sterographic coordinates) so that velocity components are in meters 
geo_to_polar = Transformer.from_crs("EPSG:4326", "EPSG:3031", always_xy=True)

def proj_geo_to_polar(lat, lon):
    east, north = geo_to_polar.transform(lon, lat)
    return east, north


# Looked at what the columns were using describe in command line to set these 

column_labels = [
    "dir", "frame", "stn", "doy", "date", "time", #direction, reference frame, station ID, day of year, date (yyyy-mm-dd), time
    "nsv", "gdop", "rmsc", "rmsp", # number of satellietes used, geometric effects on precision, RMS of code observations, RMS of phase observations
    "dlat", "dlon", "dhgt", # displacement from reference latitude, longitude and height
    "sdlat", "sdlon", "sdhgt", # 95% confidence for lat, long, and height
    "latdd", "latmn", "latss", # latitude as degrees, minutes, and seconds 
    "londd", "lonmn", "lonss", # longitude as degrees, minutes, and seconds 
    "hgt", "utmzone", "utm_e", "utm_n", "utm_s1", "utm_s2" # ellipsoidal height, UTM (universal transverse mercator aka repping location of Earth with flat (x,y) coords) zone number, how far east and north in meters, point scale factor and combined scale factor
]

'''Function to get one of the PPP files into a cleaned data frame of GPS epochs. Uses the backward and forward basses 
as position estimates to construct the decimal lat and long from the degree-minute-seconds column and apply a filter based on the position RMS to limit noise'''

def parse_one_pos(filepath):
    rows = []
    # Only keeping lines that start with a backward or forward pass 
    with open(filepath, 'r') as f:
        for line in f:
            if line.startswith("BWD") or line.startswith("FWD"):
                parts = line.split()
                if len(parts) >= 22:
                    rows.append(parts)
    if not rows:
        return None
    
    #getting rid of columns that are not needed based on the slip catalog     
    raw_df = pd.DataFrame(rows)
    raw_df = raw_df.iloc[:, :len(column_labels)]
    raw_df.columns = column_labels[:raw_df.shape[1]]

    # Getting the timestamp from seperate date and time columns 
    raw_df["timestamp"] = pd.to_datetime(
        raw_df["date"] + " " + raw_df["time"],
        format="%Y-%m-%d %H:%M:%S.%f",
        errors="coerce"
    )

    # Numeric conversions from degree-minute-seconds 
    dms_columns = ["rmsp", "latdd", "latmn", "latss", "londd", "lonmn", "lonss", "hgt"]
    for col in dms_columns:
        if col in raw_df.columns:
            raw_df[col] = pd.to_numeric(raw_df[col], errors="coerce")

    # Going back to decimal degrees from the degree-minute-second format
    lat_sign = np.where(raw_df["latdd"] < 0, -1, 1)
    long_sign = np.where(raw_df["londd"] < 0, -1, 1)
    raw_df["lat"] = lat_sign * (raw_df["latdd"].abs() + raw_df["latmn"] / 60 + raw_df["latss"] / 3600)
    raw_df["lon"] = long_sign * (raw_df["londd"].abs() + raw_df["lonmn"] / 60 + raw_df["lonss"] / 3600)

    # Getting rid of epochs that have a lot of noise in their position based on the phase observation RMS
    raw_df = raw_df[raw_df["rmsp"] <= position_threshold].copy()
    raw_df = raw_df.dropna(subset=["timestamp", "lat", "lon"])
    raw_df = raw_df.sort_values("timestamp").reset_index(drop=True)

    return raw_df[["timestamp", "lat", "lon", "hgt", "rmsp"]]